# 00 — Checkout and provenance

Run this notebook before thesis analysis to record the software checkout and its basic
hardware-free checks. Use **Restart Kernel and Run All** from an existing Linux Python 3.10+
Jupyter environment on the development host, with Git, Docker daemon access and the existing
framework development image. Framework checks run through `scripts/dev.sh` in Docker;
the notebook kernel only orchestrates and displays them.

If the image is absent, build it explicitly from the repository root with
`./scripts/dev.sh build-image`, then rerun. This notebook never pulls/builds an image,
installs packages, resolves campaigns, signs firmware, or contacts the gateway.

Save this notebook before running. Keep the source notebook's outputs cleared before committing;
the retained report is the provenance artifact. Each execution creates a new ignored directory
under `results/checkout/`. Existing pytest hooks may clean incompatible generated build state.


In [ ]:
from pathlib import Path
import sys

# Leave as None when launched from the repository or notebooks directory.
# Otherwise set Path("/home/timk/rmit/PQC-Framework-02").
REPO_ROOT = None
start = Path(REPO_ROOT or Path.cwd()).resolve()
root = next((p for p in (start, *start.parents)
             if (p / "AI_BUILD_GUIDE.md").is_file()
             and (p / "scripts" / "checkout_report.py").is_file()), None)
if root is None:
    raise RuntimeError("Set REPO_ROOT to the PQC-Framework-02 checkout.")
sys.path.insert(0, str(root / "scripts"))
from checkout_report import run_checkout, render_markdown

print(f"Repository: {root}")


## Checks and evidence boundary

The fixed check list covers shell syntax, staged/unstaged Git whitespace checks,
immutable dependency pins, the complete unit suite, and the reporter's local-process
integration tests. It deliberately does not run the general integration/hardware suites,
firmware builds, signing, broker operations, gateway probes or device workflows.

The report records UTC timestamps, exact commands and exit codes, pytest pass/fail/error/skip
counts, branch/commit (including detached HEAD), staged/unstaged/untracked paths, recursive
submodule revisions, selected input SHA-256 values and host/container versions. Docker calls
use the inspected local image ID even if its tag changes. Missing prerequisites and missing
JUnit evidence produce an **incomplete** report; failures stay **failed** and skips stay visible.
The checks continue after an ordinary failure so the report captures the remaining outcomes.


In [ ]:
report, report_dir = run_checkout(root)


In [ ]:
from IPython.display import Markdown, display

display(Markdown(render_markdown(report)))
print(f"Retained JSON: {report_dir / 'report.json'}")
print(f"Retained Markdown: {report_dir / 'report.md'}")
print((report_dir / "report.sha256").read_text())


## Using this in thesis analysis

Retain the complete run directory. Reference `run_id`, Git commit, dirty status, command
outcomes and `report.sha256` alongside later analysis inputs. `report.json` inventories logs,
JUnit files and the Markdown report by SHA-256. From the retained directory, verify the
JSON identity with `sha256sum -c report.sha256`.

A dirty checkout cannot be reconstructed from its commit alone; status records and selected
input hashes are not a source archive. The before/after comparison detects Git status,
submodule revision and selected-input changes, not every possible concurrent source edit.
Avoid editing during a run. No patch contents, ignored files, remote URLs, credential files
or environment dumps are collected. Logs apply limited secret-pattern redaction; review logs
before sharing because arbitrary test output cannot be guaranteed secret-free.

A pass covers the listed software checks only. Skipped checks do not prove a capability.
Host durations are not firmware timing measurements. This report establishes no gateway,
UART, flashing, device runtime, memory, benchmark, network, MQTT/TLS or end-to-end result.
